# Lab Day 1 — Nguyễn Văn Hồng — 2A202602800

Notebook hoàn chỉnh cho Forest CoverType, gồm **Part 0–4**: chuẩn bị dữ liệu, kiểm tra model, baseline, thí nghiệm và đánh giá cuối. Các ô code đã được chạy theo thứ tự từ đầu đến cuối; output được giữ trong notebook.

Chạy notebook từ thư mục `submission_2A202602800/code/` để đường dẫn `../../data`, `../../scripts`, `../results` và `../figures` hoạt động đúng. Nếu chưa có `data/processed/*.npz`, Part 0 sẽ tạo chúng từ metadata cố định. Môi trường đã đo trong bài là PyTorch CPU; mixed precision CUDA chưa được thử.

Cấu hình cuối được chọn chỉ bằng validation trước khi chạy script chấm eval. Số liệu nộp nằm trong `../experiments.xlsx`, `../predictions_eval.csv`, `../eval_result.json` và `../REPORT.md`.


In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, math
import numpy as np, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO_ROOT = "../.."   # chạy notebook từ submission_2A202602800/code/
OUT_DIR = ".."
device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)
print("PyTorch:", torch.__version__, "| device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


PyTorch: 2.14.0+cpu | device: cpu


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [2]:
processed = f"{REPO_ROOT}/data/processed"
if not os.path.exists(f"{processed}/train.npz") or not os.path.exists(f"{processed}/eval.npz"):
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, env=env, check=True)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=processed)
print("Shapes:", data["X_tr"].shape, data["X_val"].shape, data["X_eval"].shape)
numeric = data["X_tr"][:, :10].cpu().numpy().astype(np.float64)
print("Numeric train mean:", np.round(numeric.mean(0), 5))
print("Numeric train std:", np.round(numeric.std(0), 5))
assert np.max(np.abs(numeric.mean(0))) < 1e-4
assert np.max(np.abs(numeric.std(0) - 1)) < 1e-4


train=371,847, val=92,962, eval=116,203
majority class=1, val accuracy=0.4876
Shapes: torch.Size([371847, 54]) torch.Size([92962, 54]) torch.Size([116203, 54])
Numeric train mean: [-0. -0.  0.  0.  0.  0. -0. -0. -0.  0.]
Numeric train std: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [3]:
# Cố định seed, kiểm tra shape, số tham số và loss trước bước cập nhật đầu tiên.
torch.manual_seed(1)
np.random.seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47_879
logits = model(torch.randn(8, 54, device=device))
assert logits.shape == (8, 7)
model.eval()
with torch.no_grad():
    loss_sum = 0.0
    for start in range(0, len(data["X_val"]), 8192):
        xb = data["X_val"][start:start + 8192]
        yb = data["y_val"][start:start + 8192]
        loss_sum += F.cross_entropy(model(xb), yb, reduction="sum").item()
step0_loss = loss_sum / len(data["y_val"])
print(f"logits={tuple(logits.shape)}, parameters={count_params(model)}")
print(f"step0 CE={step0_loss:.4f}; ln(7)={math.log(7):.4f}")
print("activation std after each Linear:", activation_stats(model, data["X_val"][:512]))

# Kiểm tra gradient và khả năng ghi nhớ 20 mẫu; đây là phép thử, không phải thí nghiệm eval.
torch.manual_seed(1)
small_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
small_model.train()
opt = torch.optim.Adam(small_model.parameters(), lr=0.003)
x20, y20 = data["X_tr"][:20], data["y_tr"][:20]
health_steps, health_losses = [], []
for step in range(601):
    opt.zero_grad(set_to_none=True)
    out = small_model(x20)
    loss = F.cross_entropy(out, y20)
    loss.backward()
    if step == 0:
        grad_norms = {name: p.grad.norm().item() for name, p in small_model.named_parameters()}
        print("grad norms:", grad_norms)
        assert all(value > 0 for value in grad_norms.values())
    acc20 = (out.argmax(1) == y20).float().mean().item()
    health_steps.append(step)
    health_losses.append(loss.item())
    if acc20 == 1.0 and loss.item() < 0.001:
        break
    opt.step()
assert acc20 == 1.0 and loss.item() < 0.01
print(f"20-sample overfit: step={step}, CE={loss.item():.6f}, accuracy={acc20:.0%}")
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(health_steps, health_losses)
ax.set(title="Overfit 20 training samples", xlabel="Update step", ylabel="Cross-entropy")
ax.grid(alpha=0.3)
fig.savefig(f"{OUT_DIR}/figures/healthcheck_20_samples.png", dpi=150, bbox_inches="tight")
plt.show()


logits=(8, 7), parameters=47879
step0 CE=2.2691; ln(7)=1.9459
activation std after each Linear: [0.6662053465843201, 0.6463871598243713, 0.5933443903923035]


grad norms: {'net.0.weight': 0.9022009372711182, 'net.0.bias': 0.3931546211242676, 'net.2.weight': 2.776932716369629, 'net.2.bias': 0.5370684266090393, 'net.4.weight': 2.4162039756774902, 'net.4.bias': 0.6395001411437988}
20-sample overfit: step=58, CE=0.000986, accuracy=100%


C:\Users\vanho\AppData\Local\Temp\ipykernel_27448\487303777.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nhận xét Part 1 (đã đo trên CPU, seed 1):** Model cho logits `(8, 7)` và có đúng 47.879 tham số. Loss bước 0 trên toàn bộ validation là **2,2691**, cao hơn mốc `ln 7 = 1,9459`: khởi tạo He tạo logits chưa bằng nhau, nên `ln 7` chỉ là mốc tham khảo chứ không phải bất biến. Gradient khác 0 ở mọi tham số. Với 20 mẫu train, loss giảm xuống dưới 0,001 và accuracy đạt 100% sau khoảng 58 bước Adam; ảnh `figures/healthcheck_20_samples.png` ghi lại đường cong. Đây là kiểm tra pipeline, chưa dùng tập eval.


## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.
**Dự đoán trước các lượt Part 2:** `pilot-s1-lr0p03` có thể học chậm hơn `lr=0,1` ở cùng 20 epoch. `base-s1` sẽ vượt mốc lớp đa số. `base-s2` và `base-s3` có thể lệch macro-F1 so với `base-s1` dù cùng cấu hình; mức lệch này dùng để ước lượng nhiễu seed.


In [4]:
# Bước 5: cùng split, seed 1 và 20 epoch để chọn lr của baseline bằng validation.
# Hai giá trị 0.03 và 0.1 là gợi ý trong GUIDE, không lấy từ eval.
# Mọi lần chạy đều lưu JSON + ảnh riêng; run đầu tiên được chọn sẽ đổi exp_id thành base-s1.
torch.set_num_threads(4)  # hạn chế tranh chấp luồng trên CPU
lr_candidates = [0.03, 0.1]
pilots = []
for lr in lr_candidates:
    candidate = {**DEFAULT_CFG, "exp_id": f"pilot-s1-lr{str(lr).replace('.', 'p')}",
                 "group": "hparam", "description": f"Baseline lr probe {lr}", "lr": lr, "seed": 1}
    result = run_experiment(candidate, data)
    pilots.append(result)
    print(candidate["exp_id"], "best_epoch=", result["summary"]["best_epoch"],
          "best_val_loss=", result["summary"]["best_val_loss"],
          "val_macro_f1=", result["summary"]["val_macro_f1"], flush=True)
    save_result(result, f"{OUT_DIR}/results")
    plot_run(result, f"{OUT_DIR}/figures/{candidate['exp_id']}.png")

# Chọn theo val loss (tiêu chí best epoch của GUIDE); không dùng tập eval.
best_pilot = min(pilots, key=lambda r: r["summary"]["best_val_loss"])
chosen_lr = best_pilot["cfg"]["lr"]
old_id = best_pilot["cfg"]["exp_id"]
os.replace(f"{OUT_DIR}/results/{old_id}.json", f"{OUT_DIR}/results/base-s1.json")
os.replace(f"{OUT_DIR}/figures/{old_id}.png", f"{OUT_DIR}/figures/base-s1.png")
best_pilot["cfg"].update(exp_id="base-s1", group="baseline", description="Baseline M-base, seed 1")
save_result(best_pilot, f"{OUT_DIR}/results")
plot_run(best_pilot, f"{OUT_DIR}/figures/base-s1.png")
plot_compare(pilots, "val_macro_f1", f"{OUT_DIR}/figures/compare_hparam_lr.png",
             title="Learning-rate probes: validation macro-F1")
print("Chosen lr by validation:", chosen_lr, flush=True)

baseline_results = [best_pilot]
for seed in (2, 3):
    cfg = {**DEFAULT_CFG, "exp_id": f"base-s{seed}", "group": "baseline",
           "description": f"Baseline M-base, seed {seed}", "lr": chosen_lr, "seed": seed}
    result = run_experiment(cfg, data)
    baseline_results.append(result)
    save_result(result, f"{OUT_DIR}/results")
    plot_run(result, f"{OUT_DIR}/figures/{cfg['exp_id']}.png")
    print(cfg["exp_id"], "best_epoch=", result["summary"]["best_epoch"],
          "best_val_loss=", result["summary"]["best_val_loss"],
          "val_macro_f1=", result["summary"]["val_macro_f1"], flush=True)

scores = np.array([r["summary"]["val_macro_f1"] for r in baseline_results])
noise_mean = scores.mean()
noise_std = scores.std(ddof=1)
print(f"Baseline val macro-F1: {noise_mean:.4f} ± {noise_std:.4f} (3 seeds); 2σ={2*noise_std:.4f}")
plot_compare(baseline_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_baseline_seeds.png",
             title="Baseline validation macro-F1 across seeds")


pilot-s1-lr0p03 best_epoch= 20 best_val_loss= 0.2604892351795542 val_macro_f1= 0.8252861050010676


pilot-s1-lr0p1 best_epoch= 20 best_val_loss= 0.22597663485294225 val_macro_f1= 0.8588063819807569


Chosen lr by validation: 0.1


base-s2 best_epoch= 19 best_val_loss= 0.22632421326640859 val_macro_f1= 0.8446661305098212


base-s3 best_epoch= 19 best_val_loss= 0.22722973003565133 val_macro_f1= 0.8566954830178383


Baseline val macro-F1: 0.8534 ± 0.0076 (3 seeds); 2σ=0.0153


**Nhận xét baseline (đã đo, 20 epoch trên CPU):** Thử learning rate `0,03` và `0,1` với cùng seed 1. `lr=0,1` được chọn **chỉ bằng val loss**: best val loss `0,2260` so với `0,2605`; val macro-F1 tương ứng `0,8588` và `0,8253`. Ba seed của baseline `lr=0,1` (`base-s1`, `base-s2`, `base-s3`) đạt val macro-F1 lần lượt `0,8588`, `0,8447`, `0,8567`: trung bình **0,8534 ± 0,0076** (độ lệch chuẩn mẫu), ngưỡng `2σ = 0,0153`. Val accuracy khoảng `0,909`, vượt rõ mốc đoán lớp đa số `0,4876`. Đường loss nhìn chung giảm; seed 3 có val loss tăng ở epoch 20, nên chọn trọng số epoch 19. Train loss cuối thấp hơn val loss, gợi ý khoảng cách tổng quát hóa nhưng chưa kết luận quá khớp mạnh. Lưu ý train loss được đo trên tập con cố định 50.000 mẫu ở chế độ `eval()`. Các thí nghiệm sau cần so chênh lệch macro-F1 với ngưỡng nhiễu này trước khi nói cấu hình nào tốt hơn.

**Đối chiếu từng lượt:** `pilot-s1-lr0p03` đạt val macro-F1 **0,8253**, thấp hơn `base-s1` **0,8588**, đúng hướng dự đoán. `base-s2` đạt **0,8447**, thấp hơn seed 1 **0,0141**; `base-s3` đạt **0,8567**, thấp hơn seed 1 **0,0021**. Các chênh lệch seed này cho thấy không nên kết luận từ một lần chạy. `base-s1` có val accuracy **0,9094**, vượt mốc lớp đa số **0,4876**.


## Part 3 — Thí nghiệm đã thực hiện

Dự đoán trước khi chạy được ghi ở ô tiếp theo. Các cấu hình trong danh sách được huấn luyện 20 epoch trên cùng train/validation split; mỗi lần chạy lưu JSON và ảnh riêng. Nhận xét sau khi chạy so với baseline và ngưỡng nhiễu seed. Sáu chủ đề đã thử: loss, optimizer, hyperparameter, dropout, clipping và khởi tạo. Mixed precision CUDA chưa chạy trên môi trường PyTorch CPU này.


### Part 3 — Dự đoán trước khi chạy

Mọi cấu hình dưới đây giữ nguyên split, seed 1, 20 epoch và M-base so với `base-s1`, trừ yếu tố ghi trong tên. `lr=0,1` của baseline đã được chọn ở Part 2. Mỗi kết luận sẽ so với độ nhiễu baseline `2σ ≈ 0,0153` macro-F1.

- **Loss (`loss-mse`)**: MSE trên logit và one-hot có thể hội tụ chậm hơn CE; sẽ so macro-F1/accuracy, không so trị số loss vì khác thang.
- **Optimizer (`opt-*`)**: Adam/AdamW có thể học nhanh ở lr nhỏ, SGD thường cần lr lớn hơn. Mỗi optimizer được thử hai lr; Adam và AdamW với weight decay 0 dự kiến gần tương đương.
- **Batch (`batch-2048`)**: batch lớn giảm số bước cập nhật mỗi epoch xuống khoảng 1/4, nên có thể học chậm hơn ở cùng lr và 20 epoch; thời gian mỗi epoch có thể thay đổi.
- **Dropout (`drop-0p3`)**: nếu khoảng train–val loss baseline nhỏ, dropout 0,3 có thể làm giảm macro-F1 do tăng chính quy hóa khi chưa cần.
- **Clipping (`clip-0p3`)**: ngưỡng 0,3 thấp hơn grad norm trung bình baseline khoảng 0,58 nên sẽ kích hoạt. Ở lr cao 0,5, clipping có thể giảm dao động so với cùng lr không clip. Hai lượt lr cao là cặp phản chứng; khi so trực tiếp với baseline phải ghi rõ đã đổi cả lr.
- **Khởi tạo (`init-xavier`, `init-zeros`)**: Xavier có thể gần He trên mạng nông này; zeros làm các nơ-ron ẩn đối xứng và ReLU tại 0, nên chất lượng dự kiến thấp.
- **Mixed precision:** PyTorch hiện là bản CPU (`torch.cuda.is_available() == False`), nên không có phép so FP32/FP16/BF16 trên cùng GPU theo GUIDE. Đây là giới hạn thiết bị của lần nộp này, không phải kết luận rằng AMP nhanh hay chậm.

**Dự đoán cụ thể cho từng `exp_id` trước lượt chạy:**

- `loss-mse`: macro-F1 thấp hơn `base-s1`.
- `opt-sgd-lr0p03`: học chậm, thấp hơn `opt-sgd-lr0p1` và baseline momentum.
- `opt-sgd-lr0p1`: cao hơn SGD 0,03 nhưng thấp hơn baseline momentum.
- `opt-adam-lr0p001`: hội tụ tốt nhưng chậm hơn Adam 0,003 sau 20 epoch.
- `opt-adam-lr0p003`: có thể đạt macro-F1 cao hơn Adam 0,001 và xấp xỉ baseline hoặc hơn nhẹ.
- `opt-adamw-lr0p001`: gần trùng Adam 0,001 vì weight decay bằng 0.
- `opt-adamw-lr0p003`: gần trùng Adam 0,003 vì weight decay bằng 0.
- `batch-2048`: macro-F1 thấp hơn batch 512 ở cùng 20 epoch do ít bước cập nhật hơn.
- `drop-0p3`: macro-F1 giảm nếu baseline chưa quá khớp rõ.
- `clip-0p3`: ngưỡng 0,3 thấp hơn grad norm baseline, nên có thể cản trở học.
- `highlr-0p5`: lr gấp 5 baseline có thể dao động hoặc giảm chất lượng.
- `highlr-0p5-clip-0p3`: ở cùng lr cao, clipping có thể bớt dao động so với không clip.
- `init-xavier`: gần He nhưng kích hoạt bước 0 nhỏ hơn, có thể giảm nhẹ F1.
- `init-zeros`: gần như không học được và đoán lớp đa số.


In [5]:
# Mọi lần chạy đều lưu ngay để tránh mất kết quả khi phiên CPU bị gián đoạn.
# Baseline đã có: SGD+momentum lr=0.1; probe lr=0.03 cũng đã có trong results/.
base = {**DEFAULT_CFG, "lr": chosen_lr, "seed": 1}
specs = [
    ("loss-mse", "loss", "MSE on logits vs one-hot", {"loss": "mse"}),
    ("opt-sgd-lr0p03", "optimizer", "SGD lr 0.03", {"optimizer": "sgd", "lr": 0.03}),
    ("opt-sgd-lr0p1", "optimizer", "SGD lr 0.1", {"optimizer": "sgd", "lr": 0.1}),
    ("opt-adam-lr0p001", "optimizer", "Adam lr 0.001", {"optimizer": "adam", "lr": 0.001}),
    ("opt-adam-lr0p003", "optimizer", "Adam lr 0.003", {"optimizer": "adam", "lr": 0.003}),
    ("opt-adamw-lr0p001", "optimizer", "AdamW lr 0.001 wd 0", {"optimizer": "adamw", "lr": 0.001}),
    ("opt-adamw-lr0p003", "optimizer", "AdamW lr 0.003 wd 0", {"optimizer": "adamw", "lr": 0.003}),
    ("batch-2048", "hparam", "Batch 2048 vs 512", {"batch": 2048}),
    ("drop-0p3", "dropout", "Dropout 0.3 vs 0", {"dropout": 0.3}),
    ("clip-0p3", "clipping", "Clip at norm 0.3 vs no clip", {"clip_norm": 0.3}),
    ("highlr-0p5", "clipping", "High lr 0.5, no clip", {"lr": 0.5}),
    ("highlr-0p5-clip-0p3", "clipping", "High lr 0.5, clip norm 0.3", {"lr": 0.5, "clip_norm": 0.3}),
    ("init-xavier", "init", "Xavier vs He", {"init": "xavier"}),
    ("init-zeros", "init", "Zero weights vs He", {"init": "zeros"}),
]
experiment_results = []
for exp_id, group, description, changes in specs:
    cfg = {**base, "exp_id": exp_id, "group": group,
           "description": description, **changes}
    result = run_experiment(cfg, data)
    experiment_results.append(result)
    save_result(result, f"{OUT_DIR}/results")
    plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
    s = result["summary"]
    print(exp_id, "best_epoch=", s["best_epoch"], "val_acc=", s["val_acc"],
          "val_macro_f1=", s["val_macro_f1"], "diverged=", s["diverged"], flush=True)

# Ảnh chồng của từng nhóm hỗ trợ nhận xét trong báo cáo.
all_results = load_results(f"{OUT_DIR}/results")
for group in ("loss", "optimizer", "hparam", "dropout", "clipping", "init"):
    group_runs = [r for r in all_results if r["cfg"]["group"] == group]
    if group_runs:
        plot_compare(group_runs + [next(r for r in all_results if r["cfg"]["exp_id"] == "base-s1")],
                     "val_macro_f1", f"{OUT_DIR}/figures/compare_{group}.png",
                     title=f"Validation macro-F1: {group} vs baseline")


loss-mse best_epoch= 19 val_acc= 0.8709580258600288 val_macro_f1= 0.736029608807888 diverged= False


opt-sgd-lr0p03 best_epoch= 18 val_acc= 0.8200662636346033 val_macro_f1= 0.6638141289113592 diverged= False


opt-sgd-lr0p1 best_epoch= 18 val_acc= 0.8579634689442999 val_macro_f1= 0.7424748378425957 diverged= False


opt-adam-lr0p001 best_epoch= 18 val_acc= 0.9019276693702803 val_macro_f1= 0.8441418457417733 diverged= False


opt-adam-lr0p003 best_epoch= 18 val_acc= 0.9154278092123663 val_macro_f1= 0.869756114730477 diverged= False


...

clip-0p3 best_epoch= 19 val_acc= 0.8937092575460941 val_macro_f1= 0.8225378296193898 diverged= False


highlr-0p5 best_epoch= 20 val_acc= 0.9046599685893161 val_macro_f1= 0.8429449423943538 diverged= False


highlr-0p5-clip-0p3 best_epoch= 19 val_acc= 0.9085755469976979 val_macro_f1= 0.8461239454490554 diverged= False


init-xavier best_epoch= 17 val_acc= 0.9015941997805555 val_macro_f1= 0.8417531379450166 diverged= False


init-zeros best_epoch= 14 val_acc= 0.4875970826789441 val_macro_f1= 0.09364999018625454 diverged= False


**Đối chiếu sau khi chạy (validation, seed 1, 20 epoch; baseline `base-s1` macro-F1 = 0,8588; ngưỡng nhiễu 2σ = 0,0153):**

- **Loss:** `loss-mse` macro-F1 `0,7360`, thấp hơn CE `0,8588`. Dự đoán đúng. MSE trên *logit* và one-hot cho val loss `0,0294`, CE cho `0,2260`; **không so hai con số loss này** vì đơn vị/định nghĩa khác nhau. CE phù hợp trực tiếp với phân loại nhiều lớp và thường cho tín hiệu gradient mạnh hơn khi dự đoán sai.
- **Optimizer:** trong mỗi loại, lr lớn hơn ở cặp thử này cho macro-F1 cao hơn. SGD tốt nhất (`opt-sgd-lr0p1`) `0,7425`; SGD+momentum baseline `0,8588`; Adam và AdamW tốt nhất tại `0,003` đều `0,8698`. So với baseline, Adam cao hơn `0,0110` nhưng **chưa vượt 2σ=0,0153**; không khẳng định thắng chắc chắn. Adam và AdamW bằng nhau khi weight decay bằng 0, đúng dự đoán; chưa kiểm tra lợi ích của decay tách riêng.
- **Batch:** `batch-2048` đạt `0,8075`, thấp hơn baseline `0,0513`; dự đoán đúng. Cùng 20 epoch nhưng batch lớn có ít bước cập nhật hơn khoảng bốn lần. Thời gian mỗi epoch giảm từ khoảng `1,96s` xuống `1,33s`, song chất lượng val giảm.
- **Dropout:** `drop-0p3` đạt `0,7698`, thấp hơn baseline `0,0890`; dự đoán đúng. Khoảng val–train loss ở best epoch baseline khoảng `0,0244`, còn dropout khoảng `0,0039`, nhưng cả hai loss của dropout cao hơn; dấu hiệu tăng chính quy hóa quá mức trong cấu hình này.
- **Clipping:** `clip-0p3` ở lr baseline đạt `0,8225`, thấp hơn `0,8588`: ngưỡng `0,3` quá chặt đối với grad norm baseline quanh `0,58`. Ở cặp lr cao `0,5`, có clip đạt `0,8461`, không clip `0,8429`; chênh `0,0032` nhỏ hơn nhiễu. Không có NaN ở lượt không clip, nên cặp này **chưa chứng minh** clipping giải quyết mất ổn định. Hai lượt lr cao thay đổi lr so với baseline và chỉ nên so trực tiếp với nhau.
- **Khởi tạo:** `init-xavier` đạt `0,8418`, kém He `0,0171` trong một seed; chênh chỉ nhỉnh hơn ngưỡng 2σ, nên cần thêm seed trước khi kết luận vững. `init-zeros` đạt `0,0936`, accuracy `0,4876` (chỉ đoán lớp đa số). Trọng số 0 giữ các neuron ẩn đối xứng; tại ReLU(0), gradient về các lớp ẩn mất tác dụng.
- **Mixed precision:** không chạy vì môi trường hiện tại chỉ có PyTorch CPU; không suy luận tăng tốc AMP trên GPU từ các số đo CPU.

**Đối chiếu theo từng `exp_id`:**

- `loss-mse`: **0,7360** — thấp hơn baseline; đúng dự đoán.
- `opt-sgd-lr0p03`: **0,6638** — thấp hơn `opt-sgd-lr0p1`; đúng dự đoán.
- `opt-sgd-lr0p1`: **0,7425** — cao hơn SGD 0,03 nhưng kém momentum; đúng dự đoán.
- `opt-adam-lr0p001`: **0,8441** — thấp hơn Adam 0,003; đúng dự đoán.
- `opt-adam-lr0p003`: **0,8698** — hơn baseline 0,0110, chưa vượt ngưỡng nhiễu; khớp hướng dự đoán nhưng chưa kết luận thắng.
- `opt-adamw-lr0p001`: **0,8441** — trùng Adam 0,001; đúng dự đoán.
- `opt-adamw-lr0p003`: **0,8698** — trùng Adam 0,003; đúng dự đoán.
- `batch-2048`: **0,8075** — thấp hơn batch 512; đúng dự đoán.
- `drop-0p3`: **0,7698** — thấp hơn baseline; đúng dự đoán.
- `clip-0p3`: **0,8225** — thấp hơn baseline; ngưỡng quá chặt như dự đoán.
- `highlr-0p5`: **0,8429** — thấp hơn baseline nhưng không NaN; chỉ khớp phần dự đoán giảm chất lượng.
- `highlr-0p5-clip-0p3`: **0,8461** — hơn cặp không clip 0,0032; nhỏ hơn nhiễu, chưa chứng minh giảm dao động có ích.
- `init-xavier`: **0,8418** — thấp hơn He 0,0171 ở seed 1; khớp hướng dự đoán, cần lặp seed.
- `init-zeros`: **0,0936** — gần đoán lớp đa số; đúng dự đoán.


In [6]:
# Độ lệch chuẩn kích hoạt sau mỗi Linear ở bước 0; cùng seed và cùng lô validation.
init_activation_stats = {}
for init_name in ("he", "xavier", "zeros"):
    torch.manual_seed(1)
    init_model = MLP(hidden=(256, 128), dropout=0.0, init=init_name).to(device)
    init_activation_stats[init_name] = activation_stats(init_model, data["X_val"][:512])
    print(init_name, [round(x, 4) for x in init_activation_stats[init_name]])


he [0.6662, 0.6464, 0.5933]
xavier [0.2781, 0.2203, 0.1969]
zeros [0.0, 0.0, 0.0]


## Part 4 — Chọn cấu hình bằng validation, đánh giá eval

**Quyết định đã chốt trước khi chạy script eval:** trong các cấu hình đã thử, `opt-adam-lr0p003` và `opt-adamw-lr0p003` cùng có best val loss thấp nhất `0,2128` và val macro-F1 `0,8698`. Chọn **Adam, lr=0,003, seed 1**, vì AdamW ở `weight_decay=0` cho cùng kết quả; giữ nguyên 20 epoch, M-base, He, CE, batch 512, FP32. Dùng trọng số **epoch 18** có val loss thấp nhất. Baseline để đối chiếu là `base-s1` (SGD+momentum, lr=0,1, seed 1, best epoch 20). Chênh val macro-F1 `0,0110` dưới ngưỡng nhiễu baseline `2σ=0,0153`, nên chưa khẳng định cấu hình cuối tốt hơn một cách vững chắc. Quyết định này dựa hoàn toàn vào validation; không thay đổi sau khi nhìn eval.


In [7]:
# Huấn luyện lại đúng cấu hình đã chốt để lấy best_state trong RAM (JSON không lưu checkpoint).
from pathlib import Path
import tempfile

repo_path = Path(REPO_ROOT).resolve()
out_path = Path(OUT_DIR).resolve()
torch.set_num_threads(4)
cfg_base_eval = {**DEFAULT_CFG, "exp_id": "base-s1", "lr": 0.1, "seed": 1}
cfg_final = {**DEFAULT_CFG, "exp_id": "opt-adam-lr0p003", "group": "optimizer",
             "description": "Adam lr 0.003", "optimizer": "adam", "lr": 0.003, "seed": 1}
result_base_eval = run_experiment(cfg_base_eval, data)
result_final = run_experiment(cfg_final, data)
for result in (result_base_eval, result_final):
    previous = json.loads((out_path / "results" / f"{result['cfg']['exp_id']}.json").read_text(encoding="utf-8"))
    assert abs(result["summary"]["best_val_loss"] - previous["summary"]["best_val_loss"]) < 1e-4
    assert result["summary"]["best_epoch"] == previous["summary"]["best_epoch"]
    print(result["cfg"]["exp_id"], "reproduced best val loss:", result["summary"]["best_val_loss"])

eval_archive = out_path / "results" / "eval"
eval_archive.mkdir(parents=True, exist_ok=True)
env = dict(os.environ, PYTHONIOENCODING="utf-8")
with tempfile.TemporaryDirectory() as temp_dir:
    baseline_pred = Path(temp_dir) / "baseline_predictions.csv"
    final_eval(cfg_base_eval, result_base_eval, data, str(baseline_pred))
    cmd_base = [sys.executable, str(repo_path / "scripts" / "evaluate.py"),
                "--pred", str(baseline_pred), "--out", str(eval_archive / "base-s1_eval.json")]
    print(subprocess.run(cmd_base, cwd=repo_path, env=env, check=True,
                         capture_output=True, text=True, encoding="utf-8").stdout)

final_pred = out_path / "predictions_eval.csv"
final_eval(cfg_final, result_final, data, str(final_pred))
cmd_final = [sys.executable, str(repo_path / "scripts" / "evaluate.py"),
             "--pred", str(final_pred), "--out", str(out_path / "eval_result.json")]
print(subprocess.run(cmd_final, cwd=repo_path, env=env, check=True,
                     capture_output=True, text=True, encoding="utf-8").stdout)
baseline_eval_scores = json.loads((eval_archive / "base-s1_eval.json").read_text(encoding="utf-8"))
final_eval_scores = json.loads((out_path / "eval_result.json").read_text(encoding="utf-8"))
print("Final minus baseline eval macro-F1:",
      round(final_eval_scores["macro_f1"] - baseline_eval_scores["macro_f1"], 6))


base-s1 reproduced best val loss: 0.22597663485294225
opt-adam-lr0p003 reproduced best val loss: 0.21277583594094562


n_eval = 116203
accuracy = 0.9086
macro_f1 = 0.8586   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9184  0.8907  0.9044
  1    56661     0.9085  0.9387  0.9233
  2     7151     0.9070  0.8965  0.9018
  3      549     0.8410  0.7614  0.7992
  4     1899     0.8032  0.7051  0.7510
  5     3473     0.8378  0.7999  0.8184
  6     4102     0.9210  0.9037  0.9123

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  37739   4300     5    0    59     8   257
1   2974  53185    90    0   251   100    61
2      0    297  6411   60    14   369     0
3      0      0    79  418     0    52     0
4     19    497    35    0  1339     9     0
5      6    219   448   19     3  2778     0
6    352     42     0    0     1     0  3707

đã ghi D:\AI_THUC_CHIEN\track4\DAY1\K4-Track4-Day1-Neuralnetwork\submission_2A202602800\results\eval\base-s1_eval.json



n_eval = 116203
accuracy = 0.9136
macro_f1 = 0.8698   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9029  0.9216  0.9122
  1    56661     0.9333  0.9171  0.9251
  2     7151     0.8954  0.9263  0.9106
  3      549     0.8980  0.7377  0.8100
  4     1899     0.7732  0.7936  0.7833
  5     3473     0.8547  0.8013  0.8272
  6     4102     0.9074  0.9342  0.9206

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  39048   2937     0    0    48     8   327
1   3929  51961   144    0   379   184    64
2      0    246  6624   34    13   234     0
3      0      3    99  405     0    42     0
4     31    335    21    0  1507     5     0
5      9    157   510   12     2  2783     0
6    232     38     0    0     0     0  3832

đã ghi D:\AI_THUC_CHIEN\track4\DAY1\K4-Track4-Day1-Neuralnetwork\submission_2A202602800\eval_result.json

Final minus baseline eval macro-F1: 0.011215


In [8]:
# Phân tích chỉ từ eval_result.json do script chính thức sinh ra.
per_class = final_eval_scores["per_class"]
cm = np.asarray(final_eval_scores["confusion_matrix"])
print("class | support | precision | recall | F1")
for entry in per_class:
    print(f"{entry['cls']:>5} | {entry['support']:>7} | {entry['precision']:.4f} | "
          f"{entry['recall']:.4f} | {entry['f1']:.4f}")
print("Confusion matrix (row=true, column=predicted):")
print(cm)
hardest = min(per_class, key=lambda entry: entry["f1"])
cls = hardest["cls"]
wrong = cm[cls].copy(); wrong[cls] = 0
print(f"Lowest F1: class {cls}; most often confused with class {wrong.argmax()} "
      f"({wrong.max()} samples)")


class | support | precision | recall | F1
    0 |   42368 | 0.9029 | 0.9216 | 0.9122
    1 |   56661 | 0.9333 | 0.9171 | 0.9251
    2 |    7151 | 0.8954 | 0.9263 | 0.9106
    3 |     549 | 0.8980 | 0.7377 | 0.8100
    4 |    1899 | 0.7732 | 0.7936 | 0.7833
    5 |    3473 | 0.8547 | 0.8013 | 0.8272
    6 |    4102 | 0.9074 | 0.9342 | 0.9206
Confusion matrix (row=true, column=predicted):
[[39048  2937     0     0    48     8   327]
 [ 3929 51961   144     0   379   184    64]
 [    0   246  6624    34    13   234     0]
 [    0     3    99   405     0    42     0]
 [   31   335    21     0  1507     5     0]
 [    9   157   510    12     2  2783     0]
 [  232    38     0     0     0     0  3832]]
Lowest F1: class 4; most often confused with class 1 (335 samples)


**Phân tích lỗi eval của cấu hình cuối:** macro-F1 `0,8698`, accuracy `0,9136`; baseline tương ứng `0,8586` và `0,9086`. Chênh macro-F1 `0,0112` nhỏ hơn ngưỡng nhiễu validation `2σ=0,0153`, nên đây là cải thiện đo được trên lần chạy này, **chưa đủ để khẳng định vượt nhiễu seed**. Lớp khó nhất theo F1 là **lớp 4 (Aspen)**: support `1.899`, precision `0,7732`, recall `0,7936`, F1 `0,7833`; có **335** mẫu lớp 4 bị dự đoán là lớp 1 (Lodgepole Pine). Lớp 3 hiếm hơn (`549` mẫu) cũng có recall thấp `0,7377` và hay nhầm với lớp 2 (`99` mẫu). Điều này phù hợp với sự mất cân bằng lớp và chồng lấn đặc trưng, nhưng muốn xác nhận nguyên nhân địa hình cụ thể cần phân tích thêm phân bố từng đặc trưng của các mẫu bị nhầm.


In [9]:
# Xuất một dòng cho mỗi JSON đã chạy; điểm eval chỉ có ở baseline và cấu hình cuối.
import openpyxl
results = load_results(f"{OUT_DIR}/results")
assert len(results) == 18
rows = []
for result in results:
    exp_id = result["cfg"]["exp_id"]
    scores = (baseline_eval_scores if exp_id == "base-s1" else
              final_eval_scores if exp_id == "opt-adam-lr0p003" else None)
    notes = "CPU run: peak_mem_MB unavailable. Train loss measured on fixed first 50,000 train rows in eval mode."
    if exp_id.startswith("highlr-"):
        notes += " Paired high-lr comparison; lr differs from baseline."
    if result["cfg"]["optimizer"] == "sgd_momentum":
        notes += " momentum=0.9."
    if exp_id == "loss-mse":
        notes += " MSE mean over samples and 7 logits; not directly comparable to CE loss."
    if exp_id == "init-zeros":
        notes += " Symmetry and inactive hidden ReLU; predicts majority class."
    row = to_row(result, eval_scores=scores, notes=notes)
    assert (out_path / row["figure_file"]).is_file(), row["figure_file"]
    rows.append(row)
xlsx_path = out_path / "experiments.xlsx"
write_xlsx(rows, str(repo_path / "templates" / "experiment_table_template.xlsx"), str(xlsx_path))
wb = openpyxl.load_workbook(xlsx_path)
summary_notes = {
    "baseline": "3 seeds; val macro-F1 0.8534 ± 0.0076; 2σ=0.0153.",
    "loss": "MSE yields lower macro-F1 than CE; loss values have different scales.",
    "optimizer": "Adam/AdamW 0.003: 0.8698; gain vs baseline 0.0110 < 2σ.",
    "hparam": "Batch 2048 has fewer updates per epoch and lower val F1.",
    "dropout": "Dropout 0.3 reduces val F1 in this setting.",
    "clipping": "High-lr clipped vs unclipped gap 0.0032 < 2σ; no divergence.",
    "amp": "Not run: current PyTorch installation is CPU-only.",
    "init": "Zeros collapse to majority prediction; Xavier close to He.",
    "final": "Adam lr=0.003 chosen by minimum val loss before eval.",
}
ws = wb["Summary"]
for row_index in range(2, ws.max_row + 1):
    group = ws.cell(row_index, 1).value
    if group in summary_notes:
        ws.cell(row_index, 8).value = summary_notes[group]
wb.save(xlsx_path)
print(f"Saved {xlsx_path}: {len(rows)} experiments, {len([r for r in rows if r['group']=='baseline'])} baseline seeds")
print("Eval scores entered only for base-s1 and opt-adam-lr0p003.")


Saved D:\AI_THUC_CHIEN\track4\DAY1\K4-Track4-Day1-Neuralnetwork\submission_2A202602800\experiments.xlsx: 18 experiments, 3 baseline seeds
Eval scores entered only for base-s1 and opt-adam-lr0p003.
